# CelebA manifold-certification ablations

Focused analysis of **local manifold size** and **PCA dimension**. The notebook intentionally contains only accuracy curves, a selected-sigma slice, compact result tables, and optional abstention curves. No heatmaps or best-setting search are included.

Here, `certified_accuracy` is the fraction of the complete test set that is both certified and correctly classified; abstentions therefore count against this metric.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_project_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / 'src').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Run this notebook from inside the project repository.')

ROOT = find_project_root()
ABLATION_DIR = ROOT / 'output/smile_classification/celeba/certify/pixel_manifold/ablation'
FIG_DIR = ROOT / 'output/analysis/celeba_manifold_ablation'
FIG_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_SIGMA = 0.50
PLOT_ABSTENTION = True
EXPECTED_K = [128, 256, 500, 750, 1000]
EXPECTED_D = [128, 192, 256, 384, 499]
EXPECTED_SIGMAS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00]

plt.rcParams.update({
    'font.family': 'STIXGeneral',
    'mathtext.fontset': 'stix',
    'font.size': 12,
    'axes.labelsize': 13,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'figure.dpi': 120,
})
print('Ablations:', ABLATION_DIR)
print('Figure exports:', FIG_DIR)

In [ ]:
def load_ablation_results(base_dir):
    rows = []
    for metrics_path in sorted(base_dir.glob('sigma_*/*/*/metrics.json')):
        metrics = json.loads(metrics_path.read_text())
        study = metrics_path.parent.parent.name
        smoothing = metrics.get('smoothing', {})
        if study == 'local_manifold_size':
            parameter = int(smoothing['knn_k'])
            parameter_name = 'K'
        elif study == 'pca_dimension':
            parameter = int(smoothing['pca_dim'])
            parameter_name = 'd'
        else:
            continue
        rows.append({
            'study': study,
            'variant': metrics_path.parent.name,
            'sigma': float(smoothing['sigma']),
            'parameter': parameter,
            'parameter_name': parameter_name,
            'knn_k': int(smoothing['knn_k']),
            'pca_dim': smoothing.get('pca_dim'),
            'certified_accuracy': float(metrics['certified_accuracy']),
            'abstain_rate': float(metrics['abstain_rate']),
            'mean_radius': float(metrics['mean_radius']),
            'mean_log_volume': float(metrics.get('volume', {}).get('mean_log_vol_mani_actual', np.nan)),
            'geometry_factor': float(metrics.get('volume', {}).get('mean_geometry_factor', np.nan)),
            'total_test_samples': int(metrics['total_test_samples']),
            'metrics_path': str(metrics_path.relative_to(ROOT)),
        })
    return pd.DataFrame(rows)

results = load_ablation_results(ABLATION_DIR)
if results.empty:
    raise FileNotFoundError(f'No completed ablation metrics found under {ABLATION_DIR}')

local_df = results.query("study == 'local_manifold_size'").copy()
pca_df = results.query("study == 'pca_dimension'").copy()
print(f'Loaded {len(local_df)} local-size and {len(pca_df)} PCA-dimension results.')

## Completeness and compact tables

In [ ]:
def completeness_table(frame, expected_values, value_label):
    expected = pd.MultiIndex.from_product(
        [EXPECTED_SIGMAS, expected_values], names=['sigma', value_label]
    ).to_frame(index=False)
    actual = frame[['sigma', 'parameter']].rename(columns={'parameter': value_label})
    status = expected.merge(actual.assign(completed=True), how='left', on=['sigma', value_label])
    status['completed'] = status['completed'].fillna(False).astype(bool)
    return status

local_status = completeness_table(local_df, EXPECTED_K, 'K')
pca_status = completeness_table(pca_df, EXPECTED_D, 'd')
print(f"Local-size: {local_status.completed.sum()}/{len(local_status)} completed")
print(f"PCA-dimension: {pca_status.completed.sum()}/{len(pca_status)} completed")

def accuracy_table(frame, column_name):
    if frame.empty:
        return pd.DataFrame()
    table = frame.pivot(index='sigma', columns='parameter', values='certified_accuracy')
    table.columns.name = column_name
    return (100 * table).round(2)

print('Certified accuracy (%) — local manifold size')
display(accuracy_table(local_df, 'K'))
print('Certified accuracy (%) — PCA dimension')
if pca_df.empty:
    print('PCA-dimension runs are not complete yet.')
else:
    display(accuracy_table(pca_df, 'd'))

def selected_sigma_report_table(frame, parameter_label):
    selected = frame[np.isclose(frame.sigma, SELECTED_SIGMA)].copy()
    if selected.empty:
        return pd.DataFrame()
    selected = selected.sort_values('parameter').rename(columns={'parameter': parameter_label})
    selected['Certified accuracy (%)'] = 100 * selected['certified_accuracy']
    selected['Abstention rate (%)'] = 100 * selected['abstain_rate']
    selected['ACR'] = selected['mean_radius']
    selected['Mean log manifold volume'] = selected['mean_log_volume']
    report = selected[[parameter_label, 'Certified accuracy (%)',
                       'Abstention rate (%)', 'ACR',
                       'Mean log manifold volume']]
    return report.round({
        'Certified accuracy (%)': 2, 'Abstention rate (%)': 2,
        'ACR': 4, 'Mean log manifold volume': 3,
    }).reset_index(drop=True)

print(f'Report table at sigma={SELECTED_SIGMA:g} — local manifold size')
local_report_table = selected_sigma_report_table(local_df, 'K')
display(local_report_table)
local_report_table.to_csv(FIG_DIR / f'report_table_local_size_sigma_{SELECTED_SIGMA:.2f}.csv', index=False)

print(f'Report table at sigma={SELECTED_SIGMA:g} — PCA dimension')
pca_report_table = selected_sigma_report_table(pca_df, 'd')
if pca_report_table.empty:
    print('PCA-dimension runs are not complete yet.')
else:
    display(pca_report_table)
    pca_report_table.to_csv(FIG_DIR / f'report_table_pca_dimension_sigma_{SELECTED_SIGMA:.2f}.csv', index=False)

## Certified accuracy across sigma

In [ ]:
def plot_across_sigma(frame, parameter_symbol, filename_stem, metric='certified_accuracy'):
    if frame.empty:
        print(f'No completed results for {filename_stem}.')
        return
    fig, ax = plt.subplots(figsize=(6.8, 4.4))
    for value, group in frame.groupby('parameter', sort=True):
        group = group.sort_values('sigma')
        ax.plot(group.sigma, 100 * group[metric], marker='o', linewidth=2,
                markersize=5, label=fr'${parameter_symbol}={int(value)}$')
    ylabel = 'Certified accuracy (%)' if metric == 'certified_accuracy' else 'Abstention rate (%)'
    ax.set_xlabel(r'Noise level $\sigma$')
    ax.set_ylabel(ylabel)
    ax.set_xticks(EXPECTED_SIGMAS)
    ax.grid(alpha=0.25, linestyle='--')
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    for extension in ('png', 'pdf'):
        fig.savefig(FIG_DIR / f'{filename_stem}.{extension}', dpi=300, bbox_inches='tight')
    plt.show()

plot_across_sigma(local_df, 'K', 'accuracy_vs_sigma_local_manifold_size')
plot_across_sigma(pca_df, 'd', 'accuracy_vs_sigma_pca_dimension')

## Certified accuracy at one sigma

In [ ]:
def plot_sigma_slice(frame, parameter_symbol, parameter_label, filename_stem,
                     metric='certified_accuracy'):
    if frame.empty:
        print(f'No completed results for {filename_stem}.')
        return
    selected = frame[np.isclose(frame.sigma, SELECTED_SIGMA)].sort_values('parameter')
    if selected.empty:
        available = sorted(frame.sigma.unique())
        print(f'sigma={SELECTED_SIGMA:g} is unavailable; completed sigmas: {available}')
        return
    fig, ax = plt.subplots(figsize=(6.2, 4.2))
    color = '#2166ac' if metric == 'certified_accuracy' else '#b2182b'
    ax.plot(selected.parameter, 100 * selected[metric],
            marker='o', linewidth=2.2, markersize=6, color=color)
    ax.set_xlabel(parameter_label)
    ylabel = 'Certified accuracy (%)' if metric == 'certified_accuracy' else 'Abstention rate (%)'
    ax.set_ylabel(ylabel)
    ax.set_xticks(selected.parameter)
    ax.set_title(fr'$\sigma={SELECTED_SIGMA:g}$')
    ax.grid(alpha=0.25, linestyle='--')
    fig.tight_layout()
    for extension in ('png', 'pdf'):
        fig.savefig(FIG_DIR / f'{filename_stem}_sigma_{SELECTED_SIGMA:.2f}.{extension}',
                    dpi=300, bbox_inches='tight')
    plt.show()

plot_sigma_slice(local_df, 'K', r'Number of neighbours $K$', 'accuracy_vs_k')
plot_sigma_slice(pca_df, 'd', r'PCA dimension $d$', 'accuracy_vs_pca_dimension')

## Abstention-rate curves

Abstention is reported both across sigma and at the selected sigma. Set `PLOT_ABSTENTION = False` in the setup cell only when these plots are not needed.

In [ ]:
if PLOT_ABSTENTION:
    plot_across_sigma(local_df, 'K', 'abstention_vs_sigma_local_manifold_size', metric='abstain_rate')
    plot_across_sigma(pca_df, 'd', 'abstention_vs_sigma_pca_dimension', metric='abstain_rate')
    plot_sigma_slice(local_df, 'K', r'Number of neighbours $K$',
                     'abstention_vs_k', metric='abstain_rate')
    plot_sigma_slice(pca_df, 'd', r'PCA dimension $d$',
                     'abstention_vs_pca_dimension', metric='abstain_rate')